# A424 TCN + FC dual-branch fusion

Image-only follow-up with a dilated temporal convolutional network, an FC-summary branch, and validation-selected late fusion with the locked RBF-SVM feature model. Subject-level mixed-site folds are jointly stratified by site and diagnosis. The test-fold target remains AUC 0.622.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import copy, random, warnings, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score
import torch, torch.nn as nn
from torch.utils.data import Dataset,DataLoader
warnings.filterwarnings('ignore')
ROOT=Path('/content/drive/MyDrive/ADHD200-data'); BENCH=ROOT/'fmri'/'strict_loso_benchmark'; BLM=ROOT/'fmri'/'brainlm_a424'
E2E=ROOT/'fmri'/'a424_end_to_end'; OUT=ROOT/'fmri'/'a424_tcn_fc_fusion'; OUT.mkdir(parents=True,exist_ok=True)
cohort=pd.read_csv(BENCH/'locked_primary_cohort_409_with_motion.csv',dtype={'subject_id':str}); y=cohort.label.to_numpy('int64'); site=cohort.site.astype(str).to_numpy(); strata=np.char.add(np.char.add(site,'__'),y.astype(str))
Xts=np.load(E2E/'a424_timeseries_192.npy',mmap_mode='r'); bank=np.load(BENCH/'a424_feature_bank.npz',allow_pickle=True)
Xfc=bank['fc_summary'].astype('float32'); Xspec=bank['spectral'].astype('float32')
meta=pd.read_csv(BLM/'brainlm_subject_embedding_metadata.csv',dtype={'subject_id':str}); emb=np.load(BLM/'brainlm_subject_embeddings.npy'); emap={s:i for i,s in enumerate(meta.subject_id.astype(str))}; Xblm=np.stack([emb[emap[s]] for s in cohort.subject_id.astype(str)]).astype('float32')
Xsvm=np.c_[Xfc,Xspec,Xblm].astype('float32'); DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; print(DEVICE,Xts.shape,Xfc.shape,Xsvm.shape)

In [ ]:
class DS(Dataset):
    def __init__(self,idx,F,aug=False): self.idx=np.asarray(idx); self.F=F; self.aug=aug
    def __len__(self): return len(self.idx)
    def __getitem__(self,k):
        i=self.idx[k]; z=np.array(Xts[i],dtype='float32',copy=True)
        if self.aug:
            z=np.roll(z,np.random.randint(-8,9),axis=0)
            if np.random.rand()<.5: z+=np.random.normal(0,.012,z.shape).astype('float32')
            if np.random.rand()<.4: z[:,np.random.choice(424,21,replace=False)]=0
        return torch.from_numpy(z),torch.from_numpy(self.F[i]),torch.tensor(y[i],dtype=torch.float32),int(i)
class Block(nn.Module):
    def __init__(self,d):
        super().__init__(); self.net=nn.Sequential(nn.Conv1d(64,64,5,padding=2*d,dilation=d),nn.BatchNorm1d(64),nn.GELU(),nn.Dropout(.2),nn.Conv1d(64,64,3,padding=d,dilation=d),nn.BatchNorm1d(64),nn.GELU(),nn.Dropout(.2))
    def forward(self,x): return x+self.net(x)
class DualNet(nn.Module):
    def __init__(self):
        super().__init__(); self.stem=nn.Sequential(nn.Conv1d(424,64,7,padding=3),nn.BatchNorm1d(64),nn.GELU()); self.tcn=nn.Sequential(Block(1),Block(2),Block(4)); self.attn=nn.Conv1d(64,1,1)
        self.fc=nn.Sequential(nn.Linear(848,128),nn.BatchNorm1d(128),nn.GELU(),nn.Dropout(.35),nn.Linear(128,32),nn.GELU())
        self.head=nn.Sequential(nn.Dropout(.4),nn.Linear(96,32),nn.GELU(),nn.Dropout(.25),nn.Linear(32,1))
    def forward(self,x,f):
        h=self.tcn(self.stem(x.transpose(1,2))); w=torch.softmax(self.attn(h),dim=-1); t=(h*w).sum(-1); return self.head(torch.cat([t,self.fc(f)],1)).squeeze(1)
def seed_all(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
@torch.no_grad()
def pred(m,loader):
    m.eval(); yy=[];pp=[];ii=[]
    for x,f,yb,ib in loader: pp.extend(torch.sigmoid(m(x.to(DEVICE),f.to(DEVICE))).cpu().numpy()); yy.extend(yb.numpy()); ii.extend(ib.numpy())
    return np.asarray(yy),np.asarray(pp),np.asarray(ii)
def train_net(tr,va,te,F,seed):
    seed_all(seed); m=DualNet().to(DEVICE); dl=DataLoader(DS(tr,F,True),16,shuffle=True); vl=DataLoader(DS(va,F),32); tl=DataLoader(DS(te,F),32)
    pw=(y[tr]==0).sum()/max((y[tr]==1).sum(),1); lossfn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw,dtype=torch.float32,device=DEVICE)); opt=torch.optim.AdamW(m.parameters(),lr=2e-4,weight_decay=3e-3)
    best=-1; state=None; wait=0
    for epoch in range(30):
        m.train()
        for x,f,yb,_ in dl:
            opt.zero_grad(); loss=lossfn(m(x.to(DEVICE),f.to(DEVICE)),yb.to(DEVICE)); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step()
        vy,vp,_=pred(m,vl); a=roc_auc_score(vy,vp)
        if a>best+1e-4: best=a; state=copy.deepcopy(m.state_dict()); wait=0
        else: wait+=1
        if wait>=7: break
    m.load_state_dict(state); return (*pred(m,vl),*pred(m,tl),best,epoch+1)

## Four-fold evaluation
FC scaling, RBF-SVM fitting, neural early stopping, and the late-fusion weight are all determined from outer-training data. The held-out test fold is never used to choose the fusion weight.

In [ ]:
rows=[]; preds=[]; outer=StratifiedKFold(4,shuffle=True,random_state=2026)
for fold,(trainval,te) in enumerate(outer.split(np.zeros(len(y)),strata),1):
    inner=StratifiedShuffleSplit(1,test_size=.15,random_state=900+fold); a,b=next(inner.split(np.zeros(len(trainval)),strata[trainval])); tr,va=trainval[a],trainval[b]
    mu=np.nanmean(Xfc[tr],0); sd=np.nanstd(Xfc[tr],0)+1e-5; F=np.nan_to_num((Xfc-mu)/sd).astype('float32')
    print('fold',fold,'train/val/test',len(tr),len(va),len(te),flush=True)
    nvy,nvp,nvi,nty,ntp,nti,vauc,epochs=train_net(tr,va,te,F,1200+fold)
    svm=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),SVC(C=.3,gamma='scale',class_weight='balanced',probability=True,random_state=42)).fit(Xsvm[tr],y[tr])
    sv=svm.predict_proba(Xsvm[va])[:,1]; st=svm.predict_proba(Xsvm[te])[:,1]
    candidates=np.linspace(0,1,11); alpha=max(candidates,key=lambda q:roc_auc_score(y[va],q*nvp+(1-q)*sv)); ep=alpha*ntp+(1-alpha)*st
    for name,p in [('tcn_fc',ntp),('rbf_svm',st),('late_fusion',ep)]:
        pr=(p>=.5).astype(int); rows.append({'model':name,'fold':fold,'auc':roc_auc_score(y[te],p),'ap':average_precision_score(y[te],p),'balanced_accuracy':balanced_accuracy_score(y[te],pr),'alpha_tcn':alpha if name=='late_fusion' else np.nan,'epochs':epochs if name=='tcn_fc' else np.nan})
        preds.extend({'model':name,'fold':fold,'subject_id':cohort.subject_id.iloc[i],'site':site[i],'y':int(y[i]),'prob':float(q)} for i,q in zip(te,p))
    print('AUC tcn/svm/fusion',*[round(rows[-3+j]['auc'],3) for j in range(3)],'alpha',alpha,flush=True)
folds=pd.DataFrame(rows); predictions=pd.DataFrame(preds); summary=(folds.groupby('model').agg(mean_auc=('auc','mean'),sd_auc=('auc','std'),mean_ap=('ap','mean'),mean_balanced_accuracy=('balanced_accuracy','mean')).reset_index().sort_values('mean_auc',ascending=False))
folds.to_csv(OUT/'tcn_fc_fusion_folds.csv',index=False); predictions.to_csv(OUT/'tcn_fc_fusion_predictions.csv',index=False); summary.to_csv(OUT/'tcn_fc_fusion_summary.csv',index=False); display(folds.round(3));display(summary.round(3));print('Locked threshold 0.622')

## Decision
A method passes only if its four-fold mean AUC exceeds 0.622. The fusion result is valid because its weight is chosen only on an inner validation split. If it does not pass, further selection against these same outer folds should stop.